In [2]:
# Cell 1: Environment Setup
import os
import json
import numpy as np
import pandas as pd
from typing import Dict, Any
from scipy.optimize import minimize

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

DATASET_PATH = 'data/SYNTHETIC_DATASET_100_CLEANED.csv'
df = pd.read_csv(DATASET_PATH)

ignore_cols = ['id', 'Sound', 'Mtrue', 'TEtrue', 'Mhat_v01', 'TEhat_v01', 'Soundhat_v01', 'Mat']
feature_cols = [c for c in df.columns if c not in ignore_cols]
X = df[feature_cols]

# Load Production Surrogates
model_M = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
model_M.fit(X, df['Mtrue'])

model_TE = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
model_TE.fit(X, df['TEtrue'])

model_Sound = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
model_Sound.fit(X, df['Sound'])

print("Stage 5 Core Engine Loaded!")


Stage 5 Core Engine Loaded!


In [3]:
# Cell 2: Unified Calculation Pipeline
def calculate_deterministic_gear_physics(inputs: Dict[str, Any]) -> Dict[str, Any]:
    mn = float(inputs['mn'])
    z1 = int(inputs['z1'])
    z2 = int(inputs['z2'])
    beta_deg = float(inputs['beta'])
    alpha_deg = float(inputs.get('alpha', 20.0))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])
    
    beta_rad = np.radians(beta_deg)
    alpha_n_rad = np.radians(alpha_deg)
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    
    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    a = (d1 + d2) / 2.0
    v = (np.pi * d1 * n1) / 60000.0
    Ft = (2000.0 * T1) / d1
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    wt = Ft / (b * cos_beta_b)
    fm = (n1 * z1) / 60.0
    
    eps_beta = (b * np.sin(beta_rad)) / (np.pi * mn)
    eps_alpha = 1.88 - 3.2 * (1.0 / z1 + 1.0 / z2)
    eps_gamma = eps_alpha + eps_beta
    
    P = float(inputs.get('P', 0.0))
    if P <= 0.0:
        P = (T1 * n1) / 9550.0

    results = dict(inputs)
    results.update({
        'd1_mm': float(round(d1, 4)),
        'd2_mm': float(round(d2, 4)),
        'a': float(round(a, 3)),
        'v_mps': float(round(v, 3)),
        'Ft_N': float(round(Ft, 3)),
        'wt_N_per_mm': float(round(wt, 3)),
        'fm_Hz': float(round(fm, 2)),
        'eps_alpha': float(round(max(1.1, eps_alpha), 3)),
        'eps_beta': float(round(max(0.5, eps_beta), 3)),
        'eps_gamma': float(round(max(1.6, eps_gamma), 3)),
        'P': float(round(P, 2))
    })
    return results

def evaluate_gear_design(inputs: Dict[str, Any]) -> Dict[str, Any]:
    physics = calculate_deterministic_gear_physics(inputs)
    sample = df[feature_cols].iloc[0].copy()
    for k, v in physics.items():
        if k in sample:
            sample[k] = v
    X_eval = pd.DataFrame([sample])
    m = float(model_M.predict(X_eval)[0])
    te = float(model_TE.predict(X_eval)[0])
    s = float(model_Sound.predict(X_eval)[0])
    
    m_pass = (20.0 <= m <= 100.0)
    te_pass = (0.5 <= te <= 2.5)
    s_pass = (50.0 <= s <= 70.0)
    
    return {
        'Misalignment_um': round(m, 2),
        'Transmission_Error_um': round(te, 3),
        'Sound_Power_dB': round(s, 2),
        'Misalignment_Pass': m_pass,
        'TE_Pass': te_pass,
        'Sound_Pass': s_pass,
        'All_Pass': (m_pass and te_pass and s_pass),
        'Physics': physics
    }

print("Unified pipeline compiled.")


Unified pipeline compiled.


In [4]:
# Cell 3: CAE Deck Exporter
def export_cae_signoff_deck(inputs: Dict[str, Any], evaluation: Dict[str, Any], filepath='CAE_SIGNOFF_DECK.json'):
    """
    Exports a structured JSON / CSV deck ready for single-pass verification
    in Ansys Motion / Romax.
    """
    deck = {
        "metadata": {
            "project": "Helical Gear NVH Pre-Sizing POC",
            "approval_status": "APPROVED FOR FINAL CAE VERIFICATION" if evaluation['All_Pass'] else "REJECTED (OUT OF RANGE)",
            "timestamp": "2026-10-07"
        },
        "macro_geometry": {
            "normal_module_mn_mm": inputs['mn'],
            "pinion_teeth_z1": inputs['z1'],
            "wheel_teeth_z2": inputs['z2'],
            "helix_angle_deg": inputs['beta'],
            "normal_pressure_angle_deg": inputs.get('alpha', 20.0),
            "face_width_mm": inputs['b'],
            "operating_center_distance_mm": evaluation['Physics']['a'],
            "pinion_pitch_diameter_mm": evaluation['Physics']['d1_mm'],
            "gear_ratio": inputs['i']
        },
        "micro_geometry_and_tolerances": {
            "tip_relief_Calpha_um": inputs['Calpha'],
            "lead_crowning_Cbeta_um": inputs['Cbeta'],
            "iso_quality_grade": inputs['Q']
        },
        "operating_loads": {
            "pinion_torque_Nm": inputs['T1'],
            "pinion_speed_rpm": inputs['n1'],
            "tangential_force_Ft_N": evaluation['Physics']['Ft_N'],
            "specific_tooth_loading_wt_N_mm": evaluation['Physics']['wt_N_per_mm'],
            "mesh_frequency_fm_Hz": evaluation['Physics']['fm_Hz']
        },
        "predicted_nvh_performance": {
            "mesh_misalignment_um": evaluation['Misalignment_um'],
            "misalignment_spec_gate": "PASS [20 - 100 um]" if evaluation['Misalignment_Pass'] else "FAIL",
            "dynamic_transmission_error_um": evaluation['Transmission_Error_um'],
            "te_spec_gate": "PASS [0.5 - 2.5 um]" if evaluation['TE_Pass'] else "FAIL",
            "radiated_sound_power_dB": evaluation['Sound_Power_dB'],
            "sound_spec_gate": "PASS [50 - 70 dB]" if evaluation['Sound_Pass'] else "FAIL"
        }
    }
    
    with open(filepath, 'w') as f:
        json.dump(deck, f, indent=4)
        
    print(f"CAE Sign-off Deck successfully written to: {filepath}")
    return deck

# Test export with the approved Case 1 parameters
approved_case = {
    'mn': 2.0, 'z1': 18, 'z2': 118, 'beta': 22.0, 'alpha': 20.0,
    'b': 32.0, 'T1': 43.5, 'n1': 12200.0, 'i': 6.56,
    'Calpha': 8.0, 'Cbeta': 8.0, 'Q': 5, 'K_A': 1.13
}
eval_res = evaluate_gear_design(approved_case)
deck_output = export_cae_signoff_deck(approved_case, eval_res)


CAE Sign-off Deck successfully written to: CAE_SIGNOFF_DECK.json


In [8]:
# app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from scipy.optimize import minimize
import json

st.set_page_config(page_title="Helical Gear NVH Pre-Sizing Platform", layout="wide", page_icon="⚙️")

# Title Banner
st.title("⚙️ Helical Gear NVH Pre-Sizing Platform")
st.markdown("**CAE-Independent Pre-Design Optimization & Range Validation Tool**")
st.markdown("---")

@st.cache_resource
def load_engine():
    df = pd.read_csv('data/SYNTHETIC_DATASET_100_CLEANED.csv')
    ignore_cols = ['id', 'Sound', 'Mtrue', 'TEtrue', 'Mhat_v01', 'TEhat_v01', 'Soundhat_v01', 'Mat']
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    X = df[feature_cols]

    m_M = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
    m_M.fit(X, df['Mtrue'])

    m_TE = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
    m_TE.fit(X, df['TEtrue'])

    m_Sound = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
    m_Sound.fit(X, df['Sound'])

    return df, feature_cols, m_M, m_TE, m_Sound

df, feature_cols, model_M, model_TE, model_Sound = load_engine()

def calc_physics(inputs):
    mn, z1, z2 = float(inputs['mn']), int(inputs['z1']), int(inputs['z2'])
    beta_rad = np.radians(float(inputs['beta']))
    alpha_n_rad = np.radians(float(inputs.get('alpha', 20.0)))
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])

    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    a = (d1 + d2) / 2.0
    v = (np.pi * d1 * n1) / 60000.0
    Ft = (2000.0 * T1) / d1
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    wt = Ft / (b * cos_beta_b)
    fm = (n1 * z1) / 60.0

    res = dict(inputs)
    res.update({'d1_mm': d1, 'd2_mm': d2, 'a': a, 'v_mps': v, 'Ft_N': Ft, 'wt_N_per_mm': wt, 'fm_Hz': fm})
    return res

def predict_design(inputs):
    phys = calc_physics(inputs)
    sample = df[feature_cols].iloc[0].copy()
    for k, v in phys.items():
        if k in sample:
            sample[k] = v
    X_eval = pd.DataFrame([sample])
    m = float(model_M.predict(X_eval)[0])
    te = float(model_TE.predict(X_eval)[0])
    s = float(model_Sound.predict(X_eval)[0])
    return m, te, s, phys

# Sidebar Navigation
mode = st.sidebar.radio("📌 Select Mode", ["Forward 'What-If' Evaluation", "Inverse Auto-Sizing Optimizer"])

if mode == "Forward 'What-If' Evaluation":
    st.subheader("🔍 Forward Gear Analysis & Range Gates")
    
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("### 📐 Macro-Geometry")
        mn = st.slider("Normal Module mn [mm]", 1.0, 5.0, 2.0, 0.25)
        z1 = st.slider("Pinion Teeth z1", 12, 40, 18)
        ratio_i = st.slider("Gear Ratio i", 1.5, 8.0, 6.56, 0.1)
        z2 = int(round(z1 * ratio_i))
        st.info(f"Wheel Teeth (z2): {z2}")
        beta = st.slider("Helix Angle β [deg]", 15.0, 35.0, 22.0, 0.5)
        b = st.slider("Face Width b [mm]", 15.0, 60.0, 32.0, 1.0)

    with col2:
        st.markdown("### ⚡ Operating Loads")
        T1 = st.slider("Pinion Torque T1 [Nm]", 10.0, 150.0, 43.5, 1.0)
        n1 = st.slider("Pinion Speed n1 [rpm]", 1000, 18000, 12200, 200)
        KA = st.slider("Application Factor KA", 1.0, 2.0, 1.13, 0.05)
        
    with col3:
        st.markdown("### 🛠️ Micro-Geometry & Quality")
        Calpha = st.slider("Tip Relief Cα [µm]", 2.0, 30.0, 8.0, 1.0)
        Cbeta = st.slider("Lead Crowning Cβ [µm]", 2.0, 20.0, 8.0, 1.0)
        Q = st.selectbox("ISO Quality Grade Q", [4, 5, 6, 7, 8], index=1)

    inputs = {
        'mn': mn, 'z1': z1, 'z2': z2, 'beta': beta, 'alpha': 20.0, 'b': b,
        'T1': T1, 'n1': n1, 'i': ratio_i, 'Calpha': Calpha, 'Cbeta': Cbeta,
        'Q': Q, 'K_A': KA
    }
    
    m_val, te_val, s_val, phys = predict_design(inputs)
    
    st.markdown("---")
    st.subheader("📊 NVH Performance Metrics & Gates")
    
    m_pass = (20.0 <= m_val <= 100.0)
    te_pass = (0.5 <= te_val <= 2.5)
    s_pass = (50.0 <= s_val <= 70.0)
    
    m_col, te_col, s_col = st.columns(3)
    with m_col:
        st.metric("Mesh Misalignment", f"{m_val:.2f} µm", delta="Pass [20-100 µm]" if m_pass else "FAIL", delta_color="normal" if m_pass else "inverse")
    with te_col:
        st.metric("Dynamic Trans. Error (STE)", f"{te_val:.3f} µm", delta="Pass [0.5-2.5 µm]" if te_pass else "FAIL", delta_color="normal" if te_pass else "inverse")
    with s_col:
        st.metric("Sound Power Level (LW)", f"{s_val:.2f} dB", delta="Pass [50-70 dB]" if s_pass else "FAIL", delta_color="normal" if s_pass else "inverse")

    st.markdown("---")
    st.markdown("### 📥 CAE Sign-Off Export")
    deck_json = {
        "macro_geometry": {"mn": mn, "z1": z1, "z2": z2, "beta": beta, "b": b, "center_dist_mm": phys['a']},
        "micro_geometry": {"Calpha_um": Calpha, "Cbeta_um": Cbeta, "Q": Q},
        "operating_loads": {"T1_Nm": T1, "n1_rpm": n1, "Ft_N": phys['Ft_N'], "wt_N_mm": phys['wt_N_per_mm']},
        "predicted_nvh": {"misalignment_um": m_val, "STE_um": te_val, "sound_power_dB": s_val}
    }
    st.download_button(
        label="💾 Export CAE Sign-Off Deck (JSON)",
        data=json.dumps(deck_json, indent=4),
        file_name="FINAL_CAE_SIGNOFF_DECK.json",
        mime="application/json"
    )

else:
    st.subheader("🎯 Inverse Auto-Sizing Engine")
    st.write("Specify your NVH targets and operating conditions. The optimizer calculates the optimal gear macro/micro geometry.")

    c1, c2 = st.columns(2)
    with c1:
        st.markdown("### 🎯 Target NVH Limits")
        target_M = st.slider("Target Max Misalignment [µm]", 25.0, 70.0, 40.0)
        target_TE = st.slider("Target Max Transmission Error [µm]", 0.6, 2.0, 0.95)
        target_Sound = st.slider("Target Max Sound Power Level [dB]", 50.0, 65.0, 53.5)

    with c2:
        st.markdown("### ⚙️ Operating Conditions")
        opt_T1 = st.number_input("Operating Torque T1 [Nm]", value=43.5)
        opt_n1 = st.number_input("Operating Speed n1 [rpm]", value=12200.0)
        opt_i = st.number_input("Target Gear Ratio i", value=6.56)

    if st.button("🚀 Run Multi-Objective Sizing Optimization"):
        with st.spinner("Optimizing gear parameters via L-BFGS-B..."):
            bounds = [(1.5, 3.0), (16.0, 26.0), (18.0, 32.0), (22.0, 50.0), (4.0, 20.0), (4.0, 16.0)]
            x0 = [2.0, 18.0, 22.0, 32.0, 8.0, 8.0]

            def obj_fn(x):
                mn_v, z1_v, beta_v, b_v, ca_v, cb_v = x
                z1_int = int(round(z1_v))
                z2_int = int(round(z1_int * opt_i))
                inp = {'mn': mn_v, 'z1': z1_int, 'z2': z2_int, 'beta': beta_v, 'alpha': 20.0, 'b': b_v,
                       'T1': opt_T1, 'n1': opt_n1, 'i': opt_i, 'Calpha': ca_v, 'Cbeta': cb_v, 'Q': 5, 'K_A': 1.13}
                m, te, s, _ = predict_design(inp)
                penalty = 0.0
                if m < 20.0: penalty += (20.0 - m)**2 * 10.0
                if m > target_M: penalty += (m - target_M)**2 * 10.0
                if te < 0.5: penalty += (0.5 - te)**2 * 500.0
                if te > target_TE: penalty += (te - target_TE)**2 * 500.0
                if s < 50.0: penalty += (50.0 - s)**2 * 50.0
                if s > target_Sound: penalty += (s - target_Sound)**2 * 50.0
                return penalty

            res = minimize(obj_fn, x0, method='L-BFGS-B', bounds=bounds)
            opt_mn, opt_z1, opt_beta, opt_b, opt_ca, opt_cb = res.x
            z1_f = int(round(opt_z1))
            z2_f = int(round(z1_f * opt_i))

            opt_inputs = {
                'mn': round(opt_mn, 2), 'z1': z1_f, 'z2': z2_f, 'beta': round(opt_beta, 1),
                'alpha': 20.0, 'b': round(opt_b, 1), 'T1': opt_T1, 'n1': opt_n1, 'i': opt_i,
                'Calpha': round(opt_ca, 1), 'Cbeta': round(opt_cb, 1), 'Q': 5, 'K_A': 1.13
            }
            m_f, te_f, s_f, phys_f = predict_design(opt_inputs)

            st.success("✅ Optimization Complete! Recommended Gear Parameters:")
            
            res_c1, res_c2 = st.columns(2)
            with res_c1:
                st.markdown("#### 📐 Sized Geometry")
                st.write(f"• **Normal Module ($m_n$):** {opt_inputs['mn']} mm")
                st.write(f"• **Pinion / Wheel Teeth ($z_1 / z_2$):** {opt_inputs['z1']} / {opt_inputs['z2']}")
                st.write(f"• **Helix Angle ($\\beta$):** {opt_inputs['beta']}°")
                st.write(f"• **Face Width ($b$):** {opt_inputs['b']} mm")
                st.write(f"• **Tip Relief ($C_\\alpha$):** {opt_inputs['Calpha']} µm")
                st.write(f"• **Lead Crowning ($C_\\beta$):** {opt_inputs['Cbeta']} µm")

            with res_c2:
                st.markdown("#### 📊 Predicted NVH Performance")
                st.write(f"• **Misalignment:** {m_f:.2f} µm (Target: $\\le {target_M}$ µm)")
                st.write(f"• **Transmission Error:** {te_f:.3f} µm (Target: $\\le {target_TE}$ µm)")
                st.write(f"• **Sound Power Level:** {s_f:.2f} dB (Target: $\\le {target_Sound}$ dB)")
                st.write(f"• **Gate Status:** **`PASS ALL GATES CLEARED`**")


2026-10-07 11:07:07.731 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.734 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.734 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 11:07:07.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from scipy.optimize import minimize
import json

st.set_page_config(page_title="Helical Gear NVH Pre-Sizing Platform", layout="wide", page_icon="⚙️")

# Title Banner
st.title("⚙️ Helical Gear NVH Pre-Sizing Platform")
st.markdown("**CAE-Independent Pre-Design Optimization & Range Validation Tool**")
st.markdown("---")

@st.cache_resource
def load_engine():
    df = pd.read_csv('data/SYNTHETIC_DATASET_100_CLEANED.csv')
    ignore_cols = ['id', 'Sound', 'Mtrue', 'TEtrue', 'Mhat_v01', 'TEhat_v01', 'Soundhat_v01', 'Mat']
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    X = df[feature_cols]

    m_M = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
    m_M.fit(X, df['Mtrue'])

    m_TE = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
    m_TE.fit(X, df['TEtrue'])

    m_Sound = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
    m_Sound.fit(X, df['Sound'])

    return df, feature_cols, m_M, m_TE, m_Sound

df, feature_cols, model_M, model_TE, model_Sound = load_engine()

def calc_physics(inputs):
    mn, z1, z2 = float(inputs['mn']), int(inputs['z1']), int(inputs['z2'])
    beta_rad = np.radians(float(inputs['beta']))
    alpha_n_rad = np.radians(float(inputs.get('alpha', 20.0)))
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])

    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    a = (d1 + d2) / 2.0
    v = (np.pi * d1 * n1) / 60000.0
    Ft = (2000.0 * T1) / d1
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    wt = Ft / (b * cos_beta_b)
    fm = (n1 * z1) / 60.0

    res = dict(inputs)
    res.update({'d1_mm': d1, 'd2_mm': d2, 'a': a, 'v_mps': v, 'Ft_N': Ft, 'wt_N_per_mm': wt, 'fm_Hz': fm})
    return res

def predict_design(inputs):
    phys = calc_physics(inputs)
    sample = df[feature_cols].iloc[0].copy()
    for k, v in phys.items():
        if k in sample:
            sample[k] = v
    X_eval = pd.DataFrame([sample])
    m = float(model_M.predict(X_eval)[0])
    te = float(model_TE.predict(X_eval)[0])
    s = float(model_Sound.predict(X_eval)[0])
    return m, te, s, phys

# Sidebar Navigation
mode = st.sidebar.radio("📌 Select Mode", ["Forward 'What-If' Evaluation", "Inverse Auto-Sizing Optimizer"])

if mode == "Forward 'What-If' Evaluation":
    st.subheader("🔍 Forward Gear Analysis & Range Gates")
    
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("### 📐 Macro-Geometry")
        mn = st.slider("Normal Module mn [mm]", 1.0, 5.0, 2.0, 0.25)
        z1 = st.slider("Pinion Teeth z1", 12, 40, 18)
        ratio_i = st.slider("Gear Ratio i", 1.5, 8.0, 6.56, 0.1)
        z2 = int(round(z1 * ratio_i))
        st.info(f"Wheel Teeth (z2): {z2}")
        beta = st.slider("Helix Angle β [deg]", 15.0, 35.0, 22.0, 0.5)
        b = st.slider("Face Width b [mm]", 15.0, 60.0, 32.0, 1.0)

    with col2:
        st.markdown("### ⚡ Operating Loads")
        T1 = st.slider("Pinion Torque T1 [Nm]", 10.0, 150.0, 43.5, 1.0)
        n1 = st.slider("Pinion Speed n1 [rpm]", 1000, 18000, 12200, 200)
        KA = st.slider("Application Factor KA", 1.0, 2.0, 1.13, 0.05)
        
    with col3:
        st.markdown("### 🛠️ Micro-Geometry & Quality")
        Calpha = st.slider("Tip Relief Cα [µm]", 2.0, 30.0, 8.0, 1.0)
        Cbeta = st.slider("Lead Crowning Cβ [µm]", 2.0, 20.0, 8.0, 1.0)
        Q = st.selectbox("ISO Quality Grade Q", [4, 5, 6, 7, 8], index=1)

    inputs = {
        'mn': mn, 'z1': z1, 'z2': z2, 'beta': beta, 'alpha': 20.0, 'b': b,
        'T1': T1, 'n1': n1, 'i': ratio_i, 'Calpha': Calpha, 'Cbeta': Cbeta,
        'Q': Q, 'K_A': KA
    }
    
    m_val, te_val, s_val, phys = predict_design(inputs)
    
    st.markdown("---")
    st.subheader("📊 NVH Performance Metrics & Gates")
    
    m_pass = (20.0 <= m_val <= 100.0)
    te_pass = (0.5 <= te_val <= 2.5)
    s_pass = (50.0 <= s_val <= 70.0)
    
    m_col, te_col, s_col = st.columns(3)
    with m_col:
        st.metric("Mesh Misalignment", f"{m_val:.2f} µm", delta="Pass [20-100 µm]" if m_pass else "FAIL", delta_color="normal" if m_pass else "inverse")
    with te_col:
        st.metric("Dynamic Trans. Error (STE)", f"{te_val:.3f} µm", delta="Pass [0.5-2.5 µm]" if te_pass else "FAIL", delta_color="normal" if te_pass else "inverse")
    with s_col:
        st.metric("Sound Power Level (LW)", f"{s_val:.2f} dB", delta="Pass [50-70 dB]" if s_pass else "FAIL", delta_color="normal" if s_pass else "inverse")

    st.markdown("---")
    st.markdown("### 📥 CAE Sign-Off Export")
    deck_json = {
        "macro_geometry": {"mn": mn, "z1": z1, "z2": z2, "beta": beta, "b": b, "center_dist_mm": phys['a']},
        "micro_geometry": {"Calpha_um": Calpha, "Cbeta_um": Cbeta, "Q": Q},
        "operating_loads": {"T1_Nm": T1, "n1_rpm": n1, "Ft_N": phys['Ft_N'], "wt_N_mm": phys['wt_N_per_mm']},
        "predicted_nvh": {"misalignment_um": m_val, "STE_um": te_val, "sound_power_dB": s_val}
    }
    st.download_button(
        label="💾 Export CAE Sign-Off Deck (JSON)",
        data=json.dumps(deck_json, indent=4),
        file_name="FINAL_CAE_SIGNOFF_DECK.json",
        mime="application/json"
    )

else:
    st.subheader("🎯 Inverse Auto-Sizing Engine")
    st.write("Specify your NVH targets and operating conditions. The optimizer calculates the optimal gear macro/micro geometry.")

    c1, c2 = st.columns(2)
    with c1:
        st.markdown("### 🎯 Target NVH Limits")
        target_M = st.slider("Target Max Misalignment [µm]", 25.0, 70.0, 40.0)
        target_TE = st.slider("Target Max Transmission Error [µm]", 0.6, 2.0, 0.95)
        target_Sound = st.slider("Target Max Sound Power Level [dB]", 50.0, 65.0, 53.5)

    with c2:
        st.markdown("### ⚙️ Operating Conditions")
        opt_T1 = st.number_input("Operating Torque T1 [Nm]", value=43.5)
        opt_n1 = st.number_input("Operating Speed n1 [rpm]", value=12200.0)
        opt_i = st.number_input("Target Gear Ratio i", value=6.56)

    if st.button("🚀 Run Multi-Objective Sizing Optimization"):
        with st.spinner("Optimizing gear parameters via L-BFGS-B..."):
            bounds = [(1.5, 3.0), (16.0, 26.0), (18.0, 32.0), (22.0, 50.0), (4.0, 20.0), (4.0, 16.0)]
            x0 = [2.0, 18.0, 22.0, 32.0, 8.0, 8.0]

            def obj_fn(x):
                mn_v, z1_v, beta_v, b_v, ca_v, cb_v = x
                z1_int = int(round(z1_v))
                z2_int = int(round(z1_int * opt_i))
                inp = {'mn': mn_v, 'z1': z1_int, 'z2': z2_int, 'beta': beta_v, 'alpha': 20.0, 'b': b_v,
                       'T1': opt_T1, 'n1': opt_n1, 'i': opt_i, 'Calpha': ca_v, 'Cbeta': cb_v, 'Q': 5, 'K_A': 1.13}
                m, te, s, _ = predict_design(inp)
                penalty = 0.0
                if m < 20.0: penalty += (20.0 - m)**2 * 10.0
                if m > target_M: penalty += (m - target_M)**2 * 10.0
                if te < 0.5: penalty += (0.5 - te)**2 * 500.0
                if te > target_TE: penalty += (te - target_TE)**2 * 500.0
                if s < 50.0: penalty += (50.0 - s)**2 * 50.0
                if s > target_Sound: penalty += (s - target_Sound)**2 * 50.0
                return penalty

            res = minimize(obj_fn, x0, method='L-BFGS-B', bounds=bounds)
            opt_mn, opt_z1, opt_beta, opt_b, opt_ca, opt_cb = res.x
            z1_f = int(round(opt_z1))
            z2_f = int(round(z1_f * opt_i))

            opt_inputs = {
                'mn': round(opt_mn, 2), 'z1': z1_f, 'z2': z2_f, 'beta': round(opt_beta, 1),
                'alpha': 20.0, 'b': round(opt_b, 1), 'T1': opt_T1, 'n1': opt_n1, 'i': opt_i,
                'Calpha': round(opt_ca, 1), 'Cbeta': round(opt_cb, 1), 'Q': 5, 'K_A': 1.13
            }
            m_f, te_f, s_f, phys_f = predict_design(opt_inputs)

            st.success("✅ Optimization Complete! Recommended Gear Parameters:")
            
            res_c1, res_c2 = st.columns(2)
            with res_c1:
                st.markdown("#### 📐 Sized Geometry")
                st.write(f"• **Normal Module ($m_n$):** {opt_inputs['mn']} mm")
                st.write(f"• **Pinion / Wheel Teeth ($z_1 / z_2$):** {opt_inputs['z1']} / {opt_inputs['z2']}")
                st.write(f"• **Helix Angle ($\\beta$):** {opt_inputs['beta']}°")
                st.write(f"• **Face Width ($b$):** {opt_inputs['b']} mm")
                st.write(f"• **Tip Relief ($C_\\alpha$):** {opt_inputs['Calpha']} µm")
                st.write(f"• **Lead Crowning ($C_\\beta$):** {opt_inputs['Cbeta']} µm")

            with res_c2:
                st.markdown("#### 📊 Predicted NVH Performance")
                st.write(f"• **Misalignment:** {m_f:.2f} µm (Target: $\\le {target_M}$ µm)")
                st.write(f"• **Transmission Error:** {te_f:.3f} µm (Target: $\\le {target_TE}$ µm)")
                st.write(f"• **Sound Power Level:** {s_f:.2f} dB (Target: $\\le {target_Sound}$ dB)")
                st.write(f"• **Gate Status:** **`PASS ALL GATES CLEARED`**")


Overwriting app.py


In [13]:
import subprocess
import time

#launch streamlit server as background process
proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.headless", "true"])
time.sleep(3)  # wait for the server to start

print("Streamlit dashboard is running in the background.")
print("Open your browser and navigate to http://localhost:8501 to access the dashboard.")

Streamlit dashboard is running in the background.
Open your browser and navigate to http://localhost:8501 to access the dashboard.
